# tndrr AI internship assignment: all three tasks
Run on a Colab T4 GPU (Runtime → Run all).

In [ ]:
!nvidia-smi
!pip -q uninstall -y torchao
!pip -q install peft sentence-transformers
!apt-get -qq update > /dev/null && apt-get -qq install -y poppler-utils zstd lshw > /dev/null
!mkdir -p task1 task2/data task3

## Task 1: LoRA fine-tuning (Qwen2.5-0.5B-Instruct, smart-home command → JSON)

In [ ]:
%%writefile task1/make_data.py
"""Generate a synthetic "smart-home command -> JSON" dataset.

Each example maps a plain-English request to a fixed JSON schema:
  {"action": ..., "device": ..., "room": ..., "value": ...}

The test set is deliberately harder than train: it also uses phrasings,
synonyms and number words that never appear in the training data, so we
measure generalisation and not just memorisation.
"""
import json
import random

random.seed(42)

ACTIONS = ["turn_on", "turn_off", "set", "increase", "decrease"]
DEVICES = ["light", "fan", "ac", "heater", "tv", "speaker", "blinds"]
ROOMS = ["living_room", "bedroom", "kitchen", "office", "bathroom", None]

# surface forms: (seen in train, held out for test only)
DEVICE_WORDS = {
    "light":   (["light", "lights", "lamp"], ["bulb"]),
    "fan":     (["fan", "ceiling fan"], ["table fan"]),
    "ac":      (["ac", "air conditioner", "AC"], ["aircon", "cooler"]),
    "heater":  (["heater", "space heater"], ["radiator"]),
    "tv":      (["tv", "television", "TV"], ["telly"]),
    "speaker": (["speaker", "music speaker"], ["sound system"]),
    "blinds":  (["blinds", "curtains"], ["shades"]),
}
ROOM_WORDS = {
    "living_room": (["living room", "hall"], ["lounge", "sitting room"]),
    "bedroom":     (["bedroom", "master bedroom"], ["room where I sleep"]),
    "kitchen":     (["kitchen"], ["pantry area"]),
    "office":      (["office", "study"], ["home office", "workroom"]),
    "bathroom":    (["bathroom", "washroom"], ["restroom"]),
}
# which value each device takes when "set"
VALUE_RANGE = {
    "light": (10, 100), "fan": (1, 5), "ac": (16, 30), "heater": (18, 30),
    "tv": (5, 60), "speaker": (5, 80), "blinds": (0, 100),
}
NUM_WORDS = {1: "one", 2: "two", 3: "three", 4: "four", 5: "five", 10: "ten",
             16: "sixteen", 18: "eighteen", 20: "twenty", 22: "twenty two",
             24: "twenty four", 25: "twenty five", 30: "thirty", 50: "fifty"}

TEMPLATES = {
    "turn_on": (
        ["turn on the {dev}{room}", "switch on the {dev}{room}", "please turn the {dev} on{room}",
         "{dev} on{room}", "can you start the {dev}{room}"],
        ["could you fire up the {dev}{room}", "I want the {dev}{room} running"],
    ),
    "turn_off": (
        ["turn off the {dev}{room}", "switch off the {dev}{room}", "please shut the {dev} off{room}",
         "{dev} off{room}", "stop the {dev}{room}"],
        ["kill the {dev}{room}", "I don't need the {dev}{room} anymore"],
    ),
    "set": (
        ["set the {dev}{room} to {val}", "change the {dev}{room} to {val}", "put the {dev}{room} at {val}",
         "{dev}{room} to {val} please", "make the {dev}{room} {val}"],
        ["I'd like the {dev}{room} on {val}", "adjust {dev}{room} so it's at {val}"],
    ),
    "increase": (
        ["increase the {dev}{room}", "turn up the {dev}{room}", "increase the {dev}{room} by {val}",
         "raise the {dev}{room} by {val}"],
        ["bump the {dev}{room} up a bit", "crank up the {dev}{room} by {val}"],
    ),
    "decrease": (
        ["decrease the {dev}{room}", "turn down the {dev}{room}", "decrease the {dev}{room} by {val}",
         "lower the {dev}{room} by {val}"],
        ["bring the {dev}{room} down a little", "dial down the {dev}{room} by {val}"],
    ),
}


def pick(pair, held_out):
    seen, unseen = pair
    if held_out and unseen and random.random() < 0.6:
        return random.choice(unseen)
    return random.choice(seen)


def make_example(held_out=False):
    action = random.choice(ACTIONS)
    device = random.choice(DEVICES)
    room = random.choice(ROOMS)
    tmpl = pick(TEMPLATES[action], held_out)

    value = None
    val_text = ""
    if "{val}" in tmpl:
        if action == "set":
            lo, hi = VALUE_RANGE[device]
            value = random.randint(lo, hi)
        else:
            value = random.choice([1, 2, 3, 5, 10])
        # number words only appear in the test split
        if held_out and value in NUM_WORDS and random.random() < 0.5:
            val_text = NUM_WORDS[value]
        else:
            val_text = str(value)
        if action == "set" and device in ("ac", "heater") and random.random() < 0.5:
            val_text += " degrees"
        if action == "set" and device in ("light", "blinds") and random.random() < 0.5:
            val_text += "%" if random.random() < 0.5 else " percent"

    dev_text = pick(DEVICE_WORDS[device], held_out)
    room_text = ""
    if room is not None:
        r = pick(ROOM_WORDS[room], held_out)
        room_text = random.choice([f" in the {r}", f" in {r}"])

    text = tmpl.format(dev=dev_text, room=room_text, val=val_text).strip()
    if random.random() < 0.3:
        text = text[0].upper() + text[1:]
    target = {"action": action, "device": device, "room": room, "value": value}
    return {"text": text, "target": target}


def main():
    train, seen = [], set()
    while len(train) < 1000:
        ex = make_example(held_out=False)
        if ex["text"] not in seen:
            seen.add(ex["text"])
            train.append(ex)
    test = []
    while len(test) < 100:
        ex = make_example(held_out=True)
        if ex["text"] not in seen:
            seen.add(ex["text"])
            test.append(ex)
    for name, rows in [("train.jsonl", train), ("test.jsonl", test)]:
        with open(name, "w") as f:
            for r in rows:
                f.write(json.dumps(r) + "\n")
    print(f"train={len(train)} test={len(test)}")
    for r in test[:5]:
        print(r)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile task1/train_eval.py
"""LoRA fine-tune Qwen2.5-0.5B-Instruct on smart-home command -> JSON,
then compare base vs fine-tuned on the held-out test set.

Usage:  python train_eval.py            (needs train.jsonl / test.jsonl from make_data.py)
"""
import json
import math
import random
import time

import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                            "gate_proj", "up_proj", "down_proj"])
LR, EPOCHS, BATCH, MAX_LEN = 2e-4, 2, 8, 192
SEED = 0

SYSTEM = (
    "Convert the user's smart-home command into JSON with exactly these keys: "
    '"action" (one of turn_on, turn_off, set, increase, decrease), '
    '"device" (one of light, fan, ac, heater, tv, speaker, blinds), '
    '"room" (one of living_room, bedroom, kitchen, office, bathroom, or null), '
    '"value" (an integer, or null if no number is given). Output only the JSON.'
)

torch.manual_seed(SEED)
random.seed(SEED)
dev = "cuda"


def load(path):
    return [json.loads(l) for l in open(path)]


def prompt_text(tok, text):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def parse(out):
    out = out.strip()
    s, e = out.find("{"), out.rfind("}")
    if s == -1 or e == -1:
        return None
    try:
        d = json.loads(out[s:e + 1])
        return d if isinstance(d, dict) else None
    except Exception:
        return None


@torch.no_grad()
def evaluate(model, tok, rows, tag):
    model.eval()
    tok.padding_side = "left"
    preds = []
    for i in range(0, len(rows), 16):
        batch = rows[i:i + 16]
        enc = tok([prompt_text(tok, r["text"]) for r in batch], return_tensors="pt",
                  padding=True).to(dev)
        out = model.generate(**enc, max_new_tokens=64, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        for r, o in zip(batch, out):
            raw = tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True)
            preds.append({"text": r["text"], "target": r["target"], "raw": raw, "pred": parse(raw)})
    keys = ["action", "device", "room", "value"]
    n = len(preds)
    valid = sum(p["pred"] is not None for p in preds)
    exact = sum(p["pred"] is not None and all(p["pred"].get(k) == p["target"][k] for k in keys)
                for p in preds)
    field = {k: sum(p["pred"] is not None and p["pred"].get(k) == p["target"][k] for p in preds) / n
             for k in keys}
    m = {"model": tag, "n": n, "valid_json": valid / n, "exact_match": exact / n,
         **{f"acc_{k}": round(v, 3) for k, v in field.items()}}
    print(json.dumps(m))
    return m, preds


def build_train_batch(tok, rows):
    ids, labels = [], []
    for r in rows:
        p = tok(prompt_text(tok, r["text"]), add_special_tokens=False)["input_ids"]
        a = tok(json.dumps(r["target"]) + tok.eos_token, add_special_tokens=False)["input_ids"]
        x = (p + a)[:MAX_LEN]
        y = ([-100] * len(p) + a)[:MAX_LEN]  # loss only on the answer tokens
        ids.append(x)
        labels.append(y)
    L = max(map(len, ids))
    pad = tok.pad_token_id
    inp = torch.tensor([x + [pad] * (L - len(x)) for x in ids])
    lab = torch.tensor([y + [-100] * (L - len(y)) for y in labels])
    att = torch.tensor([[1] * len(x) + [0] * (L - len(x)) for x in ids])
    return inp.to(dev), att.to(dev), lab.to(dev)


def main():
    train, test = load("train.jsonl"), load("test.jsonl")
    tok = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32).to(dev)
    print("GPU:", torch.cuda.get_device_name(0))

    base_m, base_p = evaluate(model, tok, test, "base")

    model = get_peft_model(model, LoraConfig(task_type="CAUSAL_LM", **LORA))
    model.print_trainable_parameters()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR,
                            weight_decay=0.0)
    steps_total = EPOCHS * math.ceil(len(train) / BATCH)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / 20) * max(0.0, 1 - s / steps_total))
    scaler = torch.cuda.amp.GradScaler()

    torch.cuda.reset_peak_memory_stats()
    t0, step = time.time(), 0
    tok.padding_side = "right"
    for ep in range(EPOCHS):
        model.train()
        random.shuffle(train)
        for i in range(0, len(train), BATCH):
            inp, att, lab = build_train_batch(tok, train[i:i + BATCH])
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(input_ids=inp, attention_mask=att, labels=lab).loss
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad()
            sched.step()
            step += 1
            if step % 25 == 0:
                print(f"epoch {ep} step {step}/{steps_total} loss {loss.item():.4f}")
    train_time = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"train_time_s={train_time:.1f} peak_mem_GB={peak:.2f}")

    ft_m, ft_p = evaluate(model, tok, test, "lora")

    results = {"base": base_m, "lora": ft_m, "train_time_s": round(train_time, 1),
               "peak_train_mem_GB": round(peak, 2), "lora_config": LORA, "lr": LR,
               "epochs": EPOCHS, "batch": BATCH, "model": MODEL, "n_train": len(train)}
    json.dump(results, open("results.json", "w"), indent=2)
    with open("predictions.jsonl", "w") as f:
        for b, l in zip(base_p, ft_p):
            f.write(json.dumps({"text": b["text"], "target": b["target"],
                                "base_raw": b["raw"], "lora_raw": l["raw"]}) + "\n")

    print("=== RESULTS ===")
    print(json.dumps(results))
    print("=== FINE-TUNED FAILURES ===")
    for l in ft_p:
        if l["pred"] is None or any(l["pred"].get(k) != l["target"][k] for k in l["target"]):
            print(json.dumps({"text": l["text"], "target": l["target"], "pred": l["raw"]}))
    print("=== BASE SAMPLE OUTPUTS ===")
    for b in base_p[:8]:
        print(json.dumps({"text": b["text"], "target": b["target"], "pred": b["raw"]}))


if __name__ == "__main__":
    main()


In [ ]:
%cd /content/task1
!python make_data.py && python train_eval.py
%cd /content

## Task 2: RAG from scratch over the Llama 2 paper (77 pages)

In [ ]:
%%writefile task2/questions.json
[
 {"q": "How many tokens of data was Llama 2 pretrained on?", "gold": "trained on 2 trillion tokens of data"},
 {"q": "How long is the context window of Llama 2 compared to Llama 1?", "gold": "from 2048 tokens to 4096 tokens"},
 {"q": "Which attention variant did they adopt to make inference scale better?", "gold": "grouped-query attention (GQA) to improve inference scalability"},
 {"q": "How many warmup steps were used in the pretraining learning-rate schedule?", "gold": "warmup of 2000 steps"},
 {"q": "What kind of network interconnect does Meta's Research Super Cluster use?", "gold": "RSC uses NVIDIA Quantum"},
 {"q": "What were the estimated total carbon emissions for pretraining?", "gold": "539 tCO2 eq"},
 {"q": "What is the vocabulary size of the Llama 2 tokenizer?", "gold": "vocabulary size is 32k tokens"},
 {"q": "How many SFT annotations did the authors collect before they stopped?", "gold": "27,540 annotations"},
 {"q": "What batch size and sequence length were used for supervised fine-tuning?", "gold": "a batch size of 64, and a sequence length of 4096 tokens"},
 {"q": "Why do they add a margin term to the reward model's ranking loss?", "gold": "we use a large margin"},
 {"q": "Which two algorithms did they try for RLHF fine-tuning?", "gold": "we explored rlhf fine-tuning with two main algorithms"},
 {"q": "What clip threshold was used for PPO?", "gold": "ppo clip threshold of 0.2"},
 {"q": "What is Ghost Attention inspired by?", "gold": "a very simple method inspired by context"},
 {"q": "What win rate does the 70B chat model have against ChatGPT in human evaluation?", "gold": "36% and a tie rate of 31.5%"},
 {"q": "What share of pretraining documents did the toxicity classifier flag?", "gold": "0.2% of documents in our pretraining corpus"}
]


In [ ]:
%%writefile task2/rag.py
"""RAG from scratch over the Llama 2 paper (arXiv 2307.09288, 77 pages).

No LangChain / LlamaIndex: just an embedding model + NumPy (cosine search),
a hand-written BM25 for comparison, and a small LLM for the final answer.

Usage: python rag.py data/llama2.txt questions.json
"""
import json
import math
import re
import sys
from collections import Counter

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

EMB_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
LLM = "Qwen/Qwen2.5-1.5B-Instruct"
CHUNK_SIZES = [100, 400]   # words per chunk
OVERLAP = 0.2               # 20% overlap between neighbouring chunks
K = 5


def norm(s):
    return " ".join(s.split()).lower()


def chunk_words(text, size, overlap):
    words = text.split()
    step = max(1, int(size * (1 - overlap)))
    return [" ".join(words[i:i + size]) for i in range(0, len(words), step)
            if words[i:i + size]]


class DenseIndex:
    def __init__(self, model, chunks):
        self.chunks = chunks
        self.model = model
        self.E = model.encode(chunks, batch_size=64, normalize_embeddings=True,
                              show_progress_bar=False).astype(np.float32)

    def search(self, q, k):
        qv = self.model.encode([QUERY_PREFIX + q], normalize_embeddings=True)[0]
        scores = self.E @ qv                     # cosine similarity (vectors are unit length)
        top = np.argsort(-scores)[:k]
        return top, scores[top]


class BM25:
    def __init__(self, chunks, k1=1.5, b=0.75):
        self.toks = [re.findall(r"\w+", c.lower()) for c in chunks]
        self.tf = [Counter(t) for t in self.toks]
        self.dl = np.array([len(t) for t in self.toks])
        self.avgdl = self.dl.mean()
        df = Counter(w for t in self.toks for w in set(t))
        N = len(chunks)
        self.idf = {w: math.log(1 + (N - n + 0.5) / (n + 0.5)) for w, n in df.items()}
        self.k1, self.b = k1, b

    def scores(self, q):
        qt = re.findall(r"\w+", q.lower())
        s = np.zeros(len(self.toks))
        for i, tf in enumerate(self.tf):
            for w in qt:
                if w in tf:
                    f = tf[w]
                    s[i] += self.idf[w] * f * (self.k1 + 1) / (
                        f + self.k1 * (1 - self.b + self.b * self.dl[i] / self.avgdl))
        return s

    def search(self, q, k):
        s = self.scores(q)
        top = np.argsort(-s)[:k]
        return top, s[top]


def hybrid_search(dense, bm25, q, k, rrf_k=60):
    """Reciprocal-rank fusion of dense and BM25 rankings."""
    d_top, _ = dense.search(q, 50)
    b_top, _ = bm25.search(q, 50)
    score = Counter()
    for r, i in enumerate(d_top):
        score[i] += 1 / (rrf_k + r)
    for r, i in enumerate(b_top):
        score[i] += 1 / (rrf_k + r)
    top = [i for i, _ in score.most_common(k)]
    return np.array(top), None


def hit_rank(chunks, top, gold):
    g = norm(gold)
    for r, i in enumerate(top):
        if g in norm(chunks[i]):
            return r + 1
    return None


def main(doc_path, q_path):
    text = open(doc_path).read()
    qs = json.load(open(q_path))
    print(f"document words: {len(text.split())}")
    emb = SentenceTransformer(EMB_MODEL, device="cuda" if torch.cuda.is_available() else "cpu")

    summary, detail, indexes = {}, {}, {}
    for size in CHUNK_SIZES:
        chunks = chunk_words(text, size, OVERLAP)
        dense, bm25 = DenseIndex(emb, chunks), BM25(chunks)
        indexes[size] = (chunks, dense)
        for name, fn in [("dense", lambda q: dense.search(q, K)[0]),
                         ("bm25", lambda q: bm25.search(q, K)[0]),
                         ("hybrid", lambda q: hybrid_search(dense, bm25, q, K)[0])]:
            ranks = [hit_rank(chunks, fn(x["q"]), x["gold"]) for x in qs]
            r1 = sum(r is not None and r <= 1 for r in ranks) / len(qs)
            r5 = sum(r is not None for r in ranks) / len(qs)
            mrr = sum(1 / r for r in ranks if r) / len(qs)
            key = f"chunk{size}_{name}"
            summary[key] = {"n_chunks": len(chunks), "recall@1": round(r1, 3),
                            "recall@5": round(r5, 3), "MRR": round(mrr, 3)}
            detail[key] = ranks
            print(key, summary[key])

    print("=== PER-QUESTION RANK OF GOLD (None = not in top 5) ===")
    for i, x in enumerate(qs):
        print(i + 1, x["q"], {k: v[i] for k, v in detail.items()})

    # show what dense retrieval returned for misses, to analyse failures
    print("=== DENSE MISSES (top-1 chunk shown) ===")
    for size in CHUNK_SIZES:
        chunks, dense = indexes[size]
        for i, x in enumerate(qs):
            if detail[f"chunk{size}_dense"][i] is None:
                top, sc = dense.search(x["q"], K)
                print(json.dumps({"chunk_size": size, "q": x["q"], "gold": x["gold"],
                                  "top1_score": round(float(sc[0]), 3),
                                  "top1": chunks[top[0]][:300]}))

    # final answers with the LLM, using the better chunk size by dense recall@5
    best = max(CHUNK_SIZES, key=lambda s: (summary[f"chunk{s}_dense"]["recall@5"],
                                           summary[f"chunk{s}_dense"]["MRR"]))
    chunks, dense = indexes[best]
    tok = AutoTokenizer.from_pretrained(LLM)
    llm = AutoModelForCausalLM.from_pretrained(LLM, torch_dtype=torch.float16, device_map="cuda")
    print(f"=== ANSWERS (chunk size {best}, top-{K} context, {LLM}) ===")
    answers = []
    for x in qs:
        top, _ = dense.search(x["q"], K)
        ctx = "\n\n".join(f"[{j + 1}] {chunks[i]}" for j, i in enumerate(top))
        msgs = [{"role": "system", "content": "Answer the question using only the context. "
                 "Be brief. If the answer is not in the context, say you don't know."},
                {"role": "user", "content": f"Context:\n{ctx}\n\nQuestion: {x['q']}"}]
        prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        enc = tok(prompt, return_tensors="pt").to("cuda")
        out = llm.generate(**enc, max_new_tokens=80, do_sample=False)
        a = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        answers.append({"q": x["q"], "answer": a})
        print(json.dumps(answers[-1]))

    json.dump({"summary": summary, "ranks": detail, "answers": answers, "best_chunk": best},
              open("results.json", "w"), indent=2)
    print("=== SUMMARY ===")
    print(json.dumps(summary))


if __name__ == "__main__":
    main(*sys.argv[1:3])


In [ ]:
%cd /content/task2
!curl -sL -o data/llama2.pdf https://arxiv.org/pdf/2307.09288 && pdftotext data/llama2.pdf data/llama2.txt && pdfinfo data/llama2.pdf | grep Pages
!python rag.py data/llama2.txt questions.json
%cd /content

## Task 3: Ollama, fp16 vs q4_K_M (Qwen2.5-1.5B-Instruct)

In [ ]:
%%writefile task3/bench.py
"""Benchmark one model in two quantisation settings with Ollama.

Measures decode tokens/s (from Ollama's own eval_count / eval_duration),
prompt-processing speed, and GPU memory used while the model is loaded,
then saves both settings' outputs on the same 10 prompts for comparison.

Usage: python bench.py   (expects `ollama serve` running and both tags pulled)
"""
import json
import subprocess
import time
import urllib.request

SETTINGS = {
    "fp16": "qwen2.5:1.5b-instruct-fp16",
    "q4_K_M": "qwen2.5:1.5b-instruct-q4_K_M",
}
PROMPTS = [
    "Explain what a hash table is in two sentences.",
    "Write a Python function that returns the n-th Fibonacci number iteratively.",
    "What is 17 * 23? Show your working.",
    "Translate to French: 'The meeting has been moved to Thursday afternoon.'",
    "A train leaves at 9:40 and the trip takes 2 hours 35 minutes. When does it arrive?",
    "List three differences between TCP and UDP.",
    "Summarise the plot of Romeo and Juliet in one sentence.",
    "Write a haiku about monsoon rain.",
    "If all bloops are razzies and all razzies are lazzies, are all bloops lazzies? Explain.",
    "Give a JSON object with keys name, age, city for a fictional person.",
]
URL = "http://localhost:11434/api"


def post(path, body):
    req = urllib.request.Request(f"{URL}/{path}", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=600) as r:
        return json.loads(r.read())


def gpu_mem_mb():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip()
    return int(out.split("\n")[0])


def main():
    results = {}
    outputs = {}
    for name, tag in SETTINGS.items():
        # unload everything first so memory numbers are clean
        for other in SETTINGS.values():
            try:
                post("generate", {"model": other, "keep_alive": 0})
            except Exception:
                pass
        time.sleep(3)
        idle = gpu_mem_mb()
        post("generate", {"model": tag, "prompt": "hi", "stream": False,
                          "options": {"num_predict": 8}})  # warm-up / load
        loaded = gpu_mem_mb()
        ps = json.loads(urllib.request.urlopen(f"{URL}/ps").read())
        size_vram = next((m.get("size_vram") for m in ps.get("models", []) if m["name"] == tag), None)

        tps, pps, outs = [], [], []
        for p in PROMPTS:
            r = post("generate", {"model": tag, "prompt": p, "stream": False,
                                  "options": {"temperature": 0, "seed": 0, "num_predict": 256}})
            tps.append(r["eval_count"] / (r["eval_duration"] / 1e9))
            pps.append(r["prompt_eval_count"] / (r["prompt_eval_duration"] / 1e9))
            outs.append(r["response"].strip())
        peak = gpu_mem_mb()
        results[name] = {
            "tag": tag,
            "decode_tok_per_s_mean": round(sum(tps) / len(tps), 1),
            "decode_tok_per_s_min": round(min(tps), 1),
            "decode_tok_per_s_max": round(max(tps), 1),
            "prompt_tok_per_s_mean": round(sum(pps) / len(pps), 1),
            "gpu_mem_idle_MB": idle,
            "gpu_mem_loaded_MB": loaded,
            "gpu_mem_after_runs_MB": peak,
            "model_mem_delta_MB": loaded - idle,
            "ollama_size_vram_MB": round(size_vram / 1e6) if size_vram else None,
        }
        outputs[name] = outs
        print(json.dumps({name: results[name]}))

    json.dump({"results": results, "prompts": PROMPTS, "outputs": outputs},
              open("results.json", "w"), indent=2)
    print("=== OUTPUTS SIDE BY SIDE ===")
    for i, p in enumerate(PROMPTS):
        print(f"\n##### Prompt {i + 1}: {p}")
        for name in SETTINGS:
            print(f"--- {name} ---\n{outputs[name][i]}")
    print("=== SUMMARY ===")
    print(json.dumps(results))


if __name__ == "__main__":
    main()


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1; ollama --version
import subprocess, time, urllib.request
srv = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434/api/version'); print('ollama server up'); break
    except Exception:
        time.sleep(1)
!ollama pull qwen2.5:1.5b-instruct-fp16 > /dev/null 2>&1; ollama pull qwen2.5:1.5b-instruct-q4_K_M > /dev/null 2>&1; ollama list

In [ ]:
%cd /content/task3
!python bench.py
%cd /content

In [ ]:
!cat /content/task1/results.json /content/task2/results.json /content/task3/results.json > /dev/null && echo ALL_DONE